In [1]:
import os
import torch
from torch import nn
from torchvision import models
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import v2
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import matplotlib.pyplot as plt
import time
from PIL import Image
from tempfile import TemporaryDirectory

cudnn.benchmark = True
plt.ion()   

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [3]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 8

learning_rate = 0.01

epochs = 10

trainset = datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

/home/redinent/coding/trash/env/.venv/lib/python3.14/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
resnet18 = models.resnet18(weights = models.ResNet18_Weights.IMAGENET1K_V1)

model = resnet18
model.fc = nn.Linear(model.fc.in_features,10)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [5]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size

    print(correct*100)

In [6]:
for i in range(epochs):
    train_loop(trainloader,model,loss_fn,optimizer)
    print(i)

test_loop(testloader,model,loss_fn)

loss: 2.536815  [    8/50000]
loss: 2.530409  [  808/50000]
loss: 1.906956  [ 1608/50000]
loss: 2.330672  [ 2408/50000]
loss: 2.146293  [ 3208/50000]
loss: 2.569279  [ 4008/50000]
loss: 2.288597  [ 4808/50000]
loss: 2.360246  [ 5608/50000]
loss: 2.301677  [ 6408/50000]
loss: 2.218655  [ 7208/50000]
loss: 2.520206  [ 8008/50000]
loss: 1.982717  [ 8808/50000]
loss: 2.105595  [ 9608/50000]
loss: 2.198375  [10408/50000]
loss: 2.074517  [11208/50000]
loss: 1.615175  [12008/50000]
loss: 2.268814  [12808/50000]
loss: 2.127513  [13608/50000]
loss: 1.648077  [14408/50000]
loss: 1.827388  [15208/50000]
loss: 2.171890  [16008/50000]
loss: 2.001495  [16808/50000]
loss: 1.989455  [17608/50000]
loss: 1.877379  [18408/50000]
loss: 1.786762  [19208/50000]
loss: 1.710937  [20008/50000]
loss: 2.016492  [20808/50000]
loss: 1.648528  [21608/50000]
loss: 1.886934  [22408/50000]
loss: 1.944927  [23208/50000]
loss: 2.029507  [24008/50000]
loss: 1.832754  [24808/50000]
loss: 1.840247  [25608/50000]
loss: 1.71

In [7]:
test_loop(testloader,model,loss_fn)

71.99
